In [6]:
# Kaggle Cell 1: Setup and Configuration

import os
import math 
import sys
import numpy as np
import io

# --- INSTALL/IMPORT LIBRARIES ---
!pip install scikit-learn
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import Xception
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image, ImageChops
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score

# --- GLOBAL CONFIGURATION ---
IMAGE_SIZE = 299 
BATCH_SIZE = 32
PHASE1_EPOCHS = 5 # 5 epochs for the fast head-only training
PHASE2_EPOCHS = 15 # 15 epochs for the slow fine-tuning

# --- DATA PATH (Kaggle Native) ---
# ⚠️ Update this path if your Kaggle input folder is different
TRAINING_ROOT_FOLDER = '/kaggle/input/ff-colab-dataset/ff_c23_preprocessed' 

# --- MODEL V1 (ELA) FILENAME ---
# This is the one model we are building now.
checkpoint_path = "xceptionnet_ela_v1_best.h5" 

print(f"✅ Configuration Loaded. Training path: {TRAINING_ROOT_FOLDER}")


✅ Configuration Loaded. Training path: /kaggle/input/ff-colab-dataset/ff_c23_preprocessed


In [7]:
# Kaggle Cell 2: ELA Helper Functions

print("--- Defining ELA (Error Level Analysis) Functions ---")

def create_ela_image(img_pil, quality=90):
    """Generates an ELA image array from a PIL Image."""
    temp_buffer = io.BytesIO()
    # 1. Save/Reopen cycle to magnify compression artifacts
    img_pil.save(temp_buffer, format='JPEG', quality=quality)
    temp_buffer.seek(0)
    
    resaved_img = Image.open(temp_buffer)
    # 2. Calculate the difference (Original - Resaved)
    ela_img = ImageChops.difference(img_pil, resaved_img)
    
    # 3. Normalize the difference to make it visible
    extrema = ela_img.getextrema()
    if isinstance(extrema[0], (list, tuple)):
        max_diff = max([ex[1] for ex in extrema])
    else:
        max_diff = extrema[1] # Handle B&W images
    
    if max_diff == 0: max_diff = 1 # Avoid division by zero
    scale = 255.0 / max_diff
    ela_img = ela_img.point(lambda i: i * scale)
    
    return np.array(ela_img) 

def ela_preprocessing_function(numpy_image):
    """
    Keras Preprocessing Function:
    Takes a numpy array [0-255] and returns its ELA representation [0-1].
    """
    # 1. Convert numpy array from generator back to PIL Image
    img_pil = Image.fromarray(numpy_image.astype('uint8'), 'RGB') 
    
    # 2. Generate ELA image
    ela_array = create_ela_image(img_pil)
    
    # 3. Normalize the ELA array for the model [0-1]
    return ela_array / 255.0

print("✅ ELA functions defined.")


--- Defining ELA (Error Level Analysis) Functions ---
✅ ELA functions defined.


In [8]:
print("--- Initializing Data Generators with ELA ---")

# --- NEW: Added more augmentation ---
train_datagen = ImageDataGenerator(
    rotation_range=20,        
    horizontal_flip=True,
    width_shift_range=0.1,    # <-- NEW
    height_shift_range=0.1,   # <-- NEW
    shear_range=0.1,          # <-- NEW
    zoom_range=0.1,           # <-- NEW
    validation_split=0.2,
    preprocessing_function=ela_preprocessing_function # Apply ELA
) 

try:
    global train_generator, validation_generator
    train_generator = train_datagen.flow_from_directory(
        TRAINING_ROOT_FOLDER, 
        target_size=(IMAGE_SIZE, IMAGE_SIZE), 
        batch_size=BATCH_SIZE, 
        class_mode='binary', 
        subset='training',
        seed=42
    )
    validation_generator = train_datagen.flow_from_directory(
        TRAINING_ROOT_FOLDER, 
        target_size=(IMAGE_SIZE, IMAGE_SIZE), 
        batch_size=BATCH_SIZE, 
        class_mode='binary', 
        subset='validation',
        seed=42
    )
    print("\n✅ ELA Data Generators are ready. Training can begin.")
except Exception as e:
    print(f"❌ FATAL ERROR: Data loading failed. Check Kaggle input path. Details: {e}")


--- Initializing Data Generators with ELA ---
Found 55971 images belonging to 2 classes.
Found 13992 images belonging to 2 classes.

✅ ELA Data Generators are ready. Training can begin.


In [4]:
print("--- Defining Model Architecture (Xception) ---")

global model, base_model, history_phase1, model_checkpoint_callback, early_stopping_callback, reduce_lr_callback

# --- 1. MODEL DEFINITION ---
base_model = Xception(weights='imagenet', include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

# --- NEW: Define all callbacks ---
model_checkpoint_callback = ModelCheckpoint(filepath=checkpoint_path, save_best_only=True, monitor='val_loss', mode='min', verbose=1)
# NEW: Increased patience to 10
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True) 
# NEW: Add ReduceLROnPlateau
reduce_lr_callback = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1)

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x) # <-- NEW: Increased from 512 to 1024
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x) 
model = Model(inputs=base_model.input, outputs=predictions)

# --- 2. PHASE 1: HEAD TRAINING (5 Epochs) ---
# Freeze backbone
for layer in base_model.layers:
    layer.trainable = False 

model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
print("\n--- PHASE 1 START: HEAD TRAINING (5 Epochs) ---")

history_phase1 = model.fit(
    train_generator,
    epochs=PHASE1_EPOCHS, 
    validation_data=validation_generator,
    verbose=1,
    callbacks=[model_checkpoint_callback, early_stopping_callback, reduce_lr_callback] # <-- NEW: Added reduce_lr
)

print("\n✅ Phase 1 (Head Training) Complete.")


--- Defining Model Architecture (Xception) ---


I0000 00:00:1761965362.085676      79 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761965362.086456      79 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- PHASE 1 START: HEAD TRAINING (5 Epochs) ---


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/tmp/ipykernel_79/763347919.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img_pil = Image.fromarray(numpy_image.astype('uint8'), 'RGB')


Epoch 1/5


I0000 00:00:1761965377.605044     149 service.cc:148] XLA service 0x7a3cb443cdc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761965377.605858     149 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1761965377.605882     149 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1761965378.876481     149 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-11-01 02:49:43.956717: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng3{k11=0} for conv (f32[32,128,147,147]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,128,147,147]{3,2,1,0}, f32[128,128,1,1]{3,2,1,0}), window={size=1x1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0

1750/1750 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8520 - loss: 0.4367
Epoch 1: val_loss improved from inf to 0.41650, saving model to xceptionnet_ela_v1_best.h5
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 2906s 2s/step - accuracy: 0.8520 - loss: 0.4366 - val_accuracy: 0.8571 - val_loss: 0.4165 - learning_rate: 0.0010
Epoch 2/5
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8558 - loss: 0.4211
Epoch 2: val_loss did not improve from 0.41650
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 2278s 1s/step - accuracy: 0.8558 - loss: 0.4211 - val_accuracy: 0.8571 - val_loss: 0.4229 - learning_rate: 0.0010
Epoch 3/5
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 0s 985ms/step - accuracy: 0.8595 - loss: 0.4121
Epoch 3: val_loss did not improve from 0.41650
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 2220s 1s/step - accuracy: 0.8595 - loss: 0.4121 - val_accuracy: 0.8571 - val_loss: 0.4190 - learning_rate: 0.0010
Epoch 4/5
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 0s 928ms/step - accuracy: 0.8558 - loss: 0.4175
Epoch 4: val_loss did not improve from 0

In [5]:
print("\n--- PHASE 2 START: Fine-Tuning Backbone (15 Epochs) ---")

# 1. Load best weights from Phase 1
model.load_weights(checkpoint_path)

# --- NEW: Unfreeze from block 11 for deeper tuning ---
unfreeze_from_block = 'block11'
print(f"Unfreezing layers from {unfreeze_from_block} onwards...")

set_trainable = False
for layer in base_model.layers:
    if layer.name == unfreeze_from_block:
        set_trainable = True
    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False

# 3. Re-compile with a very low learning rate
model.compile(optimizer=Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])

# 4. Execute Phase 2
history_phase2 = model.fit(
    train_generator,
    epochs=PHASE1_EPOCHS + PHASE2_EPOCHS, # Total 20 epochs
    initial_epoch=PHASE1_EPOCHS, # Start counting from 5
    validation_data=validation_generator,
    verbose=1,
    callbacks=[model_checkpoint_callback, early_stopping_callback, reduce_lr_callback] # <-- NEW: Added callbacks here too
)

print("\n✅ Full Fine-Tuning Complete.")



--- PHASE 2 START: Fine-Tuning Backbone (15 Epochs) ---
Unfreezing layers from block11 onwards...


/tmp/ipykernel_79/763347919.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img_pil = Image.fromarray(numpy_image.astype('uint8'), 'RGB')


Epoch 6/20
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 0s 890ms/step - accuracy: 0.8567 - loss: 0.4184
Epoch 6: val_loss did not improve from 0.41650
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 2015s 1s/step - accuracy: 0.8567 - loss: 0.4184 - val_accuracy: 0.8571 - val_loss: 0.4166 - learning_rate: 1.0000e-05
Epoch 7/20
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 0s 886ms/step - accuracy: 0.8568 - loss: 0.4164
Epoch 7: val_loss did not improve from 0.41650
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 1996s 1s/step - accuracy: 0.8568 - loss: 0.4164 - val_accuracy: 0.8571 - val_loss: 0.4167 - learning_rate: 1.0000e-05
Epoch 8/20
1750/1750 ━━━━━━━━━━━━━━━━━━━━ 0s 882ms/step - accuracy: 0.8577 - loss: 0.4147

KeyboardInterrupt: 

In [9]:
# Kaggle Cell 6: Evaluation and Export

print("--- Calculating Final Performance Metrics ---")

# 1. Load the best saved model weights from the entire run
# checkpoint_path is loaded from Cell 1's global config
final_model = load_model(checkpoint_path)

# 2. Generate Predictions on the validation set
# validation_generator is loaded from Cell 3's global config
validation_generator.reset() 
num_steps = math.ceil(validation_generator.samples / validation_generator.batch_size)
Y_pred_prob = final_model.predict(validation_generator, steps=num_steps, verbose=1)

# 3. Calculate and Print Metrics
Y_true = validation_generator.classes 
Y_true_clipped = Y_true[:len(Y_pred_prob)]
Y_pred_class = (Y_pred_prob > 0.5).astype(int).flatten()

# --- NEW: Detailed Metrics from Confusion Matrix ---
cm = confusion_matrix(Y_true_clipped, Y_pred_class)
tn, fp, fn, tp = cm.ravel()

# Calculate rates
auc_roc = roc_auc_score(Y_true_clipped, Y_pred_prob.flatten())
precision = precision_score(Y_true_clipped, Y_pred_class) # TP / (TP + FP)
recall = recall_score(Y_true_clipped, Y_pred_class)       # TP / (TP + FN)
f1 = f1_score(Y_true_clipped, Y_pred_class)
specificity = tn / (tn + fp)                           # TN / (TN + FP)
false_positive_rate = fp / (tn + fp)                   # FP / (TN + FP)
overall_accuracy = (tp + tn) / (tp + tn + fp + fn)

print("\n----------------------------------------------------")
print("--- Final Model Performance (Xception-ELA V1) ---")
print("----------------------------------------------------")
print(f"| Overall Accuracy:   | {overall_accuracy:<19.4f} |")
print(f"| AUC-ROC Score:      | {auc_roc:<19.4f} |")
print("----------------------------------------------------")
print("Metrics for 'FAKE' class (Positive):")
print(f"| Precision (FAKE):   | {precision:<19.4f} |")
print(f"| Recall (FAKE):      | {recall:<19.4f} |")
print(f"| F1-Score (FAKE):    | {f1:<19.4f} |")
print("----------------------------------------------------")
print("Metrics for 'REAL' class (Negative):")
print(f"| Specificity (REAL): | {specificity:<19.4f} |")
print(f"| False Positive Rate:| {false_positive_rate:<19.4f} |")
print("----------------------------------------------------")
print("\nConfusion Matrix (Class 1 = FAKE, Class 0 = REAL):")
print("                 | Predicted Real | Predicted Fake |")
print(f"| Actual Real    | {tn:<15} {fp:<15} |")
print(f"| Actual Fake    | {fn:<15} {tp:<15} |")

# 4. Export the Model
final_model_save_path = "xceptionnet_ela_v1.h5" 
final_model.save(final_model_save_path)
print(f"\n✅ Final V1 (ELA) model saved as: {final_model_save_path}.")
print("Download from Kaggle Output ('/kaggle/working/').")



--- Calculating Final Performance Metrics ---


/tmp/ipykernel_79/763347919.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img_pil = Image.fromarray(numpy_image.astype('uint8'), 'RGB')


438/438 ━━━━━━━━━━━━━━━━━━━━ 435s 985ms/step


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



----------------------------------------------------
--- Final Model Performance (Xception-ELA V1) ---
----------------------------------------------------
| Overall Accuracy:   | 0.8571              |
| AUC-ROC Score:      | 0.5020              |
----------------------------------------------------
Metrics for 'FAKE' class (Positive):
| Precision (FAKE):   | 0.0000              |
| Recall (FAKE):      | 0.0000              |
| F1-Score (FAKE):    | 0.0000              |
----------------------------------------------------
Metrics for 'REAL' class (Negative):
| Specificity (REAL): | 1.0000              |
| False Positive Rate:| 0.0000              |
----------------------------------------------------

Confusion Matrix (Class 1 = FAKE, Class 0 = REAL):
                 | Predicted Real | Predicted Fake |
| Actual Real    | 11992           0               |
| Actual Fake    | 2000            0               |

✅ Final V1 (ELA) model saved as: xceptionnet_ela_v1.h5.
Download from Kaggle